#synthetic data with different scale and texture

In [ ]:
"""
Complete Scale Experiment Script
Creates ground truth, trains model, extracts coordinates, and computes accuracy
(radius-aware IoU)
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool, global_add_pool
from torch_geometric.data import Data, Batch
import os
import time
import pandas as pd
from tqdm import tqdm
import pickle
import random

# ============================================================================
# Configuration
# ============================================================================
BASE_DIR = r'D:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale'
os.makedirs(BASE_DIR, exist_ok=True)

# Original dimensions
ORIGINAL_X = 172
ORIGINAL_Y = 87
ORIGINAL_Z = 12

# For now, only original scale
SCALES = [
    ('original', 1, 1, 1),
    ('2x1y1z', 2, 1, 1),
    ('1x2y1z', 1, 2, 1),
    ('1x1y2z', 1, 1, 2),
    ('2x2y1z', 2, 2, 1),
    ('1x2y2z', 1, 2, 2),
    ('2x2y2z', 2, 2, 2),
    ('3x2y1z', 3, 2, 1),
    ('3x1y2z', 3, 1, 2),
    ('3x2y2z', 3, 2, 2)
]

TOP_K = 100
Z_FIXED = 5
MAX_RADIUS = 10            # Used for graphs, GT scoring
STEP_SIZE = 5

# Euclidean distance thresholds for accuracy calculation
EUCLIDEAN_THRESHOLDS = [5, 10, 15]  # Multiple thresholds to evaluate


# ============================================================================
# Helper Functions
# ============================================================================
def calculate_distance(x1, y1, z1, x2, y2, z2):
    """Calculate 3D Euclidean distance between two points"""
    return np.sqrt((x2 - x1)**2 + (y2 - y1)**2 + (z2 - z1)**2)


# ============================================================================
# 1. Create Ground Truth with Different Scales
# ============================================================================
def create_groundtruth(scale_name, x_scale, y_scale, z_scale, output_dir):
    """Create ground truth data with specified scale"""
    print(f"\n{'='*70}")
    print(f"Creating Ground Truth: {scale_name} (X={x_scale}x, Y={y_scale}x, Z={z_scale}x)")
    print(f"{'='*70}")

    num_channels = 4
    value_range = (0, 10)

    # Calculate scaled dimensions
    x_dim = ORIGINAL_X * x_scale
    y_dim = ORIGINAL_Y * y_scale
    z_dim = ORIGINAL_Z * z_scale
    num_values = value_range[1] - value_range[0] + 1

    print(f"  Dimensions: X={x_dim}, Y={y_dim}, Z={z_dim}")

    # Create 5D array: (C, V, Z, Y, X)
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)

    # Generate patterns (scaled)
    for i in range(0, 3):
        for z in range(z_dim):
            # Channel 0: x ranges (scaled)
            for y in range(y_dim):
                x_start = max(0, int((38+20*i) * x_scale / 1))
                x_end   = min(x_dim, int((43+20*i) * x_scale / 1))
                for x in range(x_start, x_end):
                    random_value = np.random.randint(6, 11)
                    data[0, random_value, z, y, x] = 1

            # Channel 1: y ranges (scaled)
            y_start = max(0, int((18+20*i) * y_scale / 1))
            y_end   = min(y_dim, int((23+20*i) * y_scale / 1))
            for y in range(y_start, y_end):
                for x in range(x_dim):
                    random_value = np.random.randint(6, 11)
                    data[1, random_value, z, y, x] = 1

            # Channel 2: diagonal lines with slope 1 (scaled)
            strip_width = 3 * max(x_scale, y_scale)
            c_values = [-60 * max(x_scale, y_scale), 0]
            for c in c_values:
                for y in range(y_dim):
                    for x in range(x_dim):
                        if abs(y * (y_dim/ORIGINAL_Y) - x * (x_dim/ORIGINAL_X) - c) <= strip_width:
                            random_value = np.random.randint(6, 11)
                            data[2, random_value, z, y, x] = 1

            # Channel 3: diagonal lines with slope -1 (scaled)
            d_values = [60 * max(x_scale, y_scale), 120 * max(x_scale, y_scale)]
            for d in d_values:
                for y in range(y_dim):
                    for x in range(x_dim):
                        if abs(y * (y_dim/ORIGINAL_Y) + x * (x_dim/ORIGINAL_X) - d) <= strip_width:
                            random_value = np.random.randint(6, 11)
                            data[3, random_value, z, y, x] = 1

    # Save ground truth
    filename = f'groundtruth_{scale_name}.npy'
    filepath = os.path.join(output_dir, filename)
    np.save(filepath, data)
    print(f"✓ Saved ground truth to: {filepath}")
    print(f"  Shape: {data.shape}, Size: {data.nbytes / 1024 / 1024:.2f} MB")

    return data, filepath


# ============================================================================
# 2. Create Subgraphs from Ground Truth
# ============================================================================
def create_subgraphs(data, scale_name, output_dir):
    """Create subgraphs from ground truth data"""
    print(f"\n{'='*70}")
    print(f"Creating Subgraphs: {scale_name}")
    print(f"{'='*70}")

    num_channels, num_values, z_dim, y_dim, x_dim = data.shape
    z_idx = Z_FIXED if Z_FIXED < z_dim else z_dim // 2

    print(f"  Using z={z_idx}, Dimensions: {y_dim}x{x_dim}")

    # Pre-compute intensity matrix and mask per channel
    print("  Pre-computing intensity matrix...")
    intensity_matrix = np.zeros((y_dim, x_dim, num_channels), dtype=np.float32)
    channel_mask = np.zeros((y_dim, x_dim, num_channels), dtype=bool)

    for channel in range(num_channels):
        # channel_data: (V, Y, X) at fixed z
        channel_data = data[channel, :, z_idx, :, :]
        value_indices = np.argmax(channel_data, axis=0)
        mask = channel_data.sum(axis=0) > 0
        intensity_matrix[:, :, channel] = np.where(mask, value_indices.astype(np.float32), 0.0)
        channel_mask[:, :, channel] = mask

    channel_counts = channel_mask.sum(axis=2)

    # Generate graph centers
    graph_centers = []
    for x in range(0, x_dim, STEP_SIZE):
        for y in range(0, y_dim, STEP_SIZE):
            graph_centers.append((x, y, z_idx))

    print(f"  Total graph centers: {len(graph_centers)}")

    # Create subgraphs
    all_subgraphs = []
    print("  Creating subgraphs...")

    for center_idx, (center_x, center_y, center_z) in enumerate(tqdm(graph_centers, desc="Processing")):
        x_min = max(0, int(center_x - MAX_RADIUS))
        x_max = min(x_dim, int(center_x + MAX_RADIUS) + 1)
        y_min = max(0, int(center_y - MAX_RADIUS))
        y_max = min(y_dim, int(center_y + MAX_RADIUS) + 1)

        nodes = []
        node_positions = []
        node_active_channels = []

        for y in range(y_min, y_max):
            for x in range(x_min, x_max):
                if channel_counts[y, x] > 0:
                    distance = calculate_distance(center_x, center_y, center_z, x, y, z_idx)
                    if distance <= MAX_RADIUS:
                        active_channels = np.where(channel_mask[y, x, :])[0].tolist()
                        intensities = intensity_matrix[y, x, active_channels]
                        nodes.append(intensities)
                        node_positions.append((x, y, z_idx))
                        node_active_channels.append(active_channels)

        if len(nodes) == 0:
            continue

        max_channels = max(len(channels) for channels in node_active_channels)
        padded_nodes = []
        for i, intensities in enumerate(nodes):
            active_ch = node_active_channels[i]
            num_ch = len(active_ch)
            if num_ch < max_channels:
                padded = np.zeros(max_channels, dtype=np.float32)
                padded[:num_ch] = intensities
                padded_nodes.append(padded)
            else:
                padded_nodes.append(intensities)

        node_features = np.array(padded_nodes, dtype=np.float32)
        node_positions_array = np.array(node_positions, dtype=np.int32)
        num_nodes = len(node_features)

        if num_nodes < 2:
            continue

        # Create edges
        node_positions_np = node_positions_array.astype(np.float32)
        if num_nodes < 50000:
            diff = node_positions_np[:, np.newaxis, :] - node_positions_np[np.newaxis, :, :]
            distances_matrix = np.sqrt(np.sum(diff**2, axis=2))
            edge_mask = (distances_matrix <= MAX_RADIUS) & (distances_matrix > 0)
            edge_i, edge_j = np.where(edge_mask)
        else:
            # For large graphs, use iterative approach
            edge_i, edge_j = [], []
            for i in range(num_nodes):
                for j in range(i+1, num_nodes):
                    dist = calculate_distance(
                        node_positions_array[i,0], node_positions_array[i,1], node_positions_array[i,2],
                        node_positions_array[j,0], node_positions_array[j,1], node_positions_array[j,2]
                    )
                    if 0 < dist <= MAX_RADIUS:
                        edge_i.extend([i, j])
                        edge_j.extend([j, i])
            edge_i, edge_j = np.array(edge_i), np.array(edge_j)

        if len(edge_i) == 0:
            continue

        edge_index = torch.tensor([edge_i, edge_j], dtype=torch.long)
        edge_weights_np = np.array([
            calculate_distance(
                node_positions_array[edge_i[k],0], node_positions_array[edge_i[k],1], node_positions_array[edge_i[k],2],
                node_positions_array[edge_j[k],0], node_positions_array[edge_j[k],1], node_positions_array[edge_j[k],2]
            ) for k in range(len(edge_i))
        ], dtype=np.float32)

        node_values = node_features.sum(axis=1)

        edge_attr = torch.tensor(edge_weights_np, dtype=torch.float32)
        x_tensor = torch.tensor(node_features, dtype=torch.float32)
        node_values_tensor = torch.tensor(node_values, dtype=torch.float32)

        graph = Data(
            x=x_tensor,
            edge_index=edge_index,
            edge_attr=edge_attr,
            center=(center_x, center_y, center_z),
            center_idx=center_idx,
            node_positions=[tuple(pos) for pos in node_positions_array],
            node_values=node_values_tensor,
            max_channels=max_channels
        )

        all_subgraphs.append(graph)

    # Save subgraphs
    filename = f'Subgraph_{scale_name}.pt'
    filepath = os.path.join(output_dir, filename)
    torch.save(all_subgraphs, filepath)
    print(f"✓ Saved {len(all_subgraphs)} subgraphs to: {filepath}")
    print(f"  File size: {os.path.getsize(filepath) / 1024 / 1024:.2f} MB")

    return all_subgraphs, filepath


# ============================================================================
# 3. Model Classes and Functions
# ============================================================================

def prepare_graph_for_batching(graph, target_channels=4):
    """Prepare graph for batching by padding to target_channels"""
    x = graph.x.clone()
    current_channels = x.shape[1]
    if current_channels < target_channels:
        padding = torch.zeros(x.shape[0], target_channels - current_channels, dtype=x.dtype, device=x.device)
        x = torch.cat([x, padding], dim=1)
    elif current_channels > target_channels:
        x = x[:, :target_channels]

    clean_graph = Data(x=x, edge_index=graph.edge_index.clone())
    if hasattr(graph, 'edge_attr') and graph.edge_attr is not None:
        clean_graph.edge_attr = graph.edge_attr.clone()
    if hasattr(graph, 'center'):
        clean_graph._original_center = graph.center

    return clean_graph


def graph_augment(graph, node_mask_ratio=0.1, edge_drop_ratio=0.05, target_channels=4):
    """Augment graph for contrastive learning"""
    aug_graph = prepare_graph_for_batching(graph, target_channels=target_channels)

    # Keep gt_score on augmented graph if present
    if hasattr(graph, 'gt_score'):
        aug_graph.gt_score = graph.gt_score

    num_nodes = aug_graph.x.shape[0]
    num_mask = int(num_nodes * node_mask_ratio)
    if num_mask > 0:
        mask_indices = torch.randperm(num_nodes)[:num_mask]
        aug_graph.x[mask_indices] = 0.0

    if edge_drop_ratio > 0 and aug_graph.edge_index.shape[1] > 0:
        num_edges = aug_graph.edge_index.shape[1]
        num_drop = int(num_edges * edge_drop_ratio)
        if num_drop > 0:
            keep_indices = torch.randperm(num_edges)[:(num_edges - num_drop)]
            aug_graph.edge_index = aug_graph.edge_index[:, keep_indices]
            if hasattr(aug_graph, 'edge_attr') and aug_graph.edge_attr is not None:
                aug_graph.edge_attr = aug_graph.edge_attr[keep_indices]

    return aug_graph


class ContrastiveGAT(nn.Module):
    """Graph Attention Network with Self-Supervised Contrastive Learning"""
    def __init__(self, in_channels=4, hidden_channels=64, projection_dim=32, num_heads=4, dropout=0.1, edge_dim=None):
        super(ContrastiveGAT, self).__init__()
        self.edge_dim = edge_dim
        gat_kwargs = dict(dropout=dropout)
        if self.edge_dim is not None and self.edge_dim > 0:
            gat_kwargs["edge_dim"] = self.edge_dim

        self.gat1 = GATConv(in_channels=in_channels, out_channels=hidden_channels, heads=num_heads, concat=True, **gat_kwargs)
        self.gat2 = GATConv(in_channels=hidden_channels * num_heads, out_channels=hidden_channels, heads=num_heads, concat=True, **gat_kwargs)
        self.gat3 = GATConv(in_channels=hidden_channels * num_heads, out_channels=hidden_channels, heads=1, concat=False, **gat_kwargs)

        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(hidden_channels * num_heads)
        self.norm2 = nn.LayerNorm(hidden_channels * num_heads)
        self.pool_dim = hidden_channels * 3

        self.projection = nn.Sequential(
            nn.Linear(self.pool_dim, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, projection_dim)
        )

        self.interaction_head = nn.Sequential(
            nn.Linear(self.pool_dim, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels // 2, 1)
        )

    def encode(self, x, edge_index, edge_attr=None, batch=None):
        if self.edge_dim is not None and self.edge_dim > 0 and edge_attr is not None:
            ea = edge_attr
        else:
            ea = None

        x = self.gat1(x, edge_index, ea)
        x = self.norm1(x)
        x = F.elu(x)
        x = self.dropout(x)

        x = self.gat2(x, edge_index, ea)
        x = self.norm2(x)
        x = F.elu(x)
        x = self.dropout(x)

        x = self.gat3(x, edge_index, ea)
        x = F.elu(x)

        return x

    def forward(self, x, edge_index, edge_attr=None, batch=None):
        node_emb = self.encode(x, edge_index, edge_attr, batch)

        if batch is None:
            batch = torch.zeros(node_emb.shape[0], dtype=torch.long, device=node_emb.device)

        mean_pool = global_mean_pool(node_emb, batch)
        max_pool = global_max_pool(node_emb, batch)
        sum_pool = global_add_pool(node_emb, batch)

        graph_emb = torch.cat([mean_pool, max_pool, sum_pool], dim=1)

        proj_emb = self.projection(graph_emb)
        proj_emb = F.normalize(proj_emb, dim=1)

        interaction_score = self.interaction_head(graph_emb)

        return proj_emb, interaction_score


def contrastive_loss(z1, z2, temperature=0.1):
    """Contrastive loss (InfoNCE) for self-supervised learning"""
    batch_size = z1.shape[0]
    device = z1.device

    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    sim_matrix = torch.matmul(z1, z2.T) / temperature
    labels = torch.arange(batch_size, device=device)

    loss = F.cross_entropy(sim_matrix, labels)
    loss_reverse = F.cross_entropy(sim_matrix.T, labels)

    return (loss + loss_reverse) / 2.0


def train_contrastive_model(model, graphs, device, epochs=10, batch_size=32, lr=0.01, gradient_accumulation_steps=4):
    """Train model with self-supervised contrastive learning"""
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    losses = []
    target_channels = max([g.x.shape[1] for g in graphs]) if len(graphs) > 0 else 4

    print(f"\n{'='*60}")
    print(f"Training Contrastive GAT Model")
    print(f"{'='*60}")
    print(f"  Epochs: {epochs}, Batch size: {batch_size}, LR: {lr}")
    print(f"  Total graphs: {len(graphs)}, Target channels: {target_channels}")

    if device.type == 'cuda':
        torch.cuda.empty_cache()

    for epoch in range(epochs):
        epoch_losses = []
        optimizer.zero_grad()

        shuffled_graphs = graphs.copy()
        random.shuffle(shuffled_graphs)

        for batch_idx, i in enumerate(range(0, len(shuffled_graphs), batch_size)):
            batch_graphs = shuffled_graphs[i:i+batch_size]

            aug1_graphs = [graph_augment(g, target_channels=target_channels) for g in batch_graphs]
            aug2_graphs = [graph_augment(g, target_channels=target_channels) for g in batch_graphs]

            for g in aug1_graphs + aug2_graphs:
                g.x = g.x.to(device)
                g.edge_index = g.edge_index.to(device)
                if hasattr(g, 'edge_attr') and g.edge_attr is not None:
                    g.edge_attr = g.edge_attr.to(device)

            try:
                batch1 = Batch.from_data_list(aug1_graphs)
                batch2 = Batch.from_data_list(aug2_graphs)
            except Exception as e:
                print(f"Error creating batch: {e}")
                continue

            z1, pred1 = model(batch1.x, batch1.edge_index, getattr(batch1, "edge_attr", None), batch1.batch)
            z2, _ = model(batch2.x, batch2.edge_index, getattr(batch2, "edge_attr", None), batch2.batch)

            # Contrastive loss
            loss_contrast = contrastive_loss(z1, z2, temperature=0.1)

            # Supervised regression loss on gt_score (if available)
            loss_reg = torch.tensor(0.0, device=device)
            if hasattr(batch1, "gt_score"):
                try:
                    gt_scores = batch1.gt_score.to(device).float()
                    pred_scores = pred1.view(-1)
                    # Normalize scores for stability
                    if gt_scores.std() > 0:
                        gt_scores = (gt_scores - gt_scores.mean()) / (gt_scores.std() + 1e-8)
                    if pred_scores.std() > 0:
                        pred_scores = (pred_scores - pred_scores.mean()) / (pred_scores.std() + 1e-8)
                    loss_reg = F.mse_loss(pred_scores, gt_scores)
                except:
                    pass

            # Combined loss
            loss = (loss_contrast + 0.5 * loss_reg) / gradient_accumulation_steps
            loss.backward()

            del z1, z2, batch1, batch2, aug1_graphs, aug2_graphs
            if device.type == 'cuda':
                torch.cuda.empty_cache()

            if (batch_idx + 1) % gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

            epoch_losses.append(loss.item() * gradient_accumulation_steps)

        if len(epoch_losses) > 0 and (len(shuffled_graphs) // batch_size) % gradient_accumulation_steps != 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        avg_loss = np.mean(epoch_losses) if len(epoch_losses) > 0 else 0.0
        losses.append(avg_loss)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.6f}")

    print(f"\n✓ Training complete!")
    return model, losses


# ============================================================================
# 4. Extract Coordinates using Model (predictions)
# ============================================================================
def find_max_interaction_positions(model, graphs, device, top_k=100):
    """
    Find positions with maximum interaction scores using the trained model.
    Uses the model's interaction_score, not any hand-crafted heuristic.
    """
    model.eval()
    all_scores = []

    target_channels = max([g.x.shape[1] for g in graphs]) if len(graphs) > 0 else 4

    with torch.no_grad():
        for graph in tqdm(graphs, desc="Processing graphs (model)"):
            # Prepare node features to have consistent channel dimension
            x = graph.x.clone()
            current_channels = x.shape[1]
            if current_channels < target_channels:
                padding = torch.zeros(x.shape[0], target_channels - current_channels, dtype=x.dtype, device=x.device)
                x = torch.cat([x, padding], dim=1)
            elif current_channels > target_channels:
                x = x[:, :target_channels]

            x = x.to(device)
            edge_index = graph.edge_index.to(device)
            if hasattr(graph, 'edge_attr') and graph.edge_attr is not None:
                edge_attr = graph.edge_attr.to(device)
            else:
                edge_attr = None

            # Forward pass: get model interaction score for this graph
            _, interaction_score = model(x, edge_index, edge_attr, batch=None)
            model_score = interaction_score.item()

            center = graph.center
            x_pos, y_pos, z_pos = center

            # Optional extra info for analysis
            num_nodes = graph.x.shape[0]
            num_edges = graph.edge_index.shape[1]
            num_channels = graph.x.shape[1] if hasattr(graph, 'x') else 0
            edge_density = num_edges / num_nodes if num_nodes > 0 else 0

            all_scores.append({
                'x': x_pos,
                'y': y_pos,
                'z': z_pos,
                'num_nodes': num_nodes,
                'num_edges': num_edges,
                'num_channels': num_channels,
                'edge_density': edge_density,
                'model_score': model_score
            })

    if len(all_scores) > 0:
        # Sort by model_score (descending): higher score = more important
        all_scores.sort(key=lambda x: x['model_score'], reverse=True)
        top_positions = all_scores[:top_k]
        return top_positions

    return []


# ============================================================================
# 5. Extract Ground-Truth Coordinates from Data (Option A)
# ============================================================================
def attach_gt_scores_to_graphs(data, graphs):
    """
    Compute a ground-truth score per graph center (same definition
    as extract_gt_coordinates_from_data) and store it as graph.gt_score.
    """
    num_channels, num_values, z_dim, y_dim, x_dim = data.shape

    # pattern_mask[z, y, x] = True if any channel / value > 0 at that voxel
    pattern_mask = (data > 0)          # (C, V, Z, Y, X)
    pattern_mask = pattern_mask.any(axis=1)  # (C, Z, Y, X)
    pattern_mask = pattern_mask.any(axis=0)  # (Z, Y, X)

    radius_sq = MAX_RADIUS ** 2

    for g in graphs:
        cx, cy, cz = g.center
        cx, cy, cz = int(cx), int(cy), int(cz)
        cz = max(0, min(z_dim - 1, cz))

        x_min = max(0, cx - MAX_RADIUS)
        x_max = min(x_dim, cx + MAX_RADIUS + 1)
        y_min = max(0, cy - MAX_RADIUS)
        y_max = min(y_dim, cy + MAX_RADIUS + 1)

        score = 0
        for y in range(y_min, y_max):
            dy2 = (y - cy) * (y - cy)
            for x in range(x_min, x_max):
                dx2 = (x - cx) * (x - cx)
                if dx2 + dy2 <= radius_sq:
                    if pattern_mask[cz, y, x]:
                        score += 1

        g.gt_score = float(score)

    return graphs


def extract_gt_coordinates_from_data(data, graphs, top_k=100):
    """
    Extract top coordinates from TRUE ground truth using Option A:
    A voxel is 'pattern' if ANY channel has a nonzero value.

    For each graph center, count how many pattern voxels are within radius MAX_RADIUS.
    Then pick the top_k centers with highest counts.
    """
    print("\nComputing ground-truth coordinates from data (Option A: ANY channel nonzero)...")

    num_channels, num_values, z_dim, y_dim, x_dim = data.shape

    # pattern_mask[z, y, x] = True if any channel / value > 0 at that voxel
    # data shape: (C, V, Z, Y, X)
    pattern_mask = (data > 0)        # bool (C, V, Z, Y, X)
    pattern_mask = pattern_mask.any(axis=1)  # any over values -> (C, Z, Y, X)
    pattern_mask = pattern_mask.any(axis=0)  # any over channels -> (Z, Y, X)

    all_scores = []
    radius_sq = MAX_RADIUS ** 2

    for graph in tqdm(graphs, desc="Processing graphs (GT)"):
        cx, cy, cz = graph.center
        cx, cy, cz = int(cx), int(cy), int(cz)

        # Safety clamp
        cz = max(0, min(z_dim - 1, cz))

        x_min = max(0, cx - MAX_RADIUS)
        x_max = min(x_dim, cx + MAX_RADIUS + 1)
        y_min = max(0, cy - MAX_RADIUS)
        y_max = min(y_dim, cy + MAX_RADIUS + 1)

        score = 0
        for y in range(y_min, y_max):
            dy2 = (y - cy) * (y - cy)
            for x in range(x_min, x_max):
                dx2 = (x - cx) * (x - cx)
                if dx2 + dy2 <= radius_sq:
                    if pattern_mask[cz, y, x]:
                        score += 1

        all_scores.append({
            'x': cx,
            'y': cy,
            'z': cz,
            'score': score
        })

    if len(all_scores) > 0:
        # Sort by true pattern count (descending)
        all_scores.sort(key=lambda s: s['score'], reverse=True)
        top_positions = all_scores[:top_k]
        return top_positions

    return []


# ============================================================================
# 6. Compute Accuracy (+ radius-aware IoU "Accuracy" column)
# ============================================================================

def compute_accuracy(model_coords, gt_coords, tol=MAX_RADIUS):
    """
    Compute accuracy metrics based on Euclidean distance between model and ground truth coordinates.
    For each model point, finds the minimum Euclidean distance to any GT point.
    
    Metrics computed:
      - Euclidean distance statistics (mean, median, min, max, std)
      - Precision, Recall, F1 based on threshold matching
      - IoU (Accuracy) = matches / (model_points + gt_points - matches)
    """
    # Convert to simple lists of (x, y, z)
    model_points = [(int(c['x']), int(c['y']), int(c['z'])) for c in model_coords]
    gt_points    = [(int(c['x']), int(c['y']), int(c['z'])) for c in gt_coords]

    model_count = len(model_points)
    gt_count    = len(gt_points)

    if model_count == 0 or gt_count == 0:
        return {
            'matches': 0,
            'precision': 0.0,
            'recall': 0.0,
            'f1_score': 0.0,
            'model_count': model_count,
            'gt_count': gt_count,
            'accuracy_iou': 0.0,
            'mean_euclidean_distance': 0.0,
            'median_euclidean_distance': 0.0,
            'min_euclidean_distance': 0.0,
            'max_euclidean_distance': 0.0,
            'std_euclidean_distance': 0.0,
        }

    # Compute Euclidean distances: for each model point, find minimum distance to any GT point
    euclidean_distances = []
    for mx, my, mz in model_points:
        min_dist = float('inf')
        for gx, gy, gz in gt_points:
            dist = calculate_distance(mx, my, mz, gx, gy, gz)
            if dist < min_dist:
                min_dist = dist
        if min_dist != float('inf'):
            euclidean_distances.append(min_dist)
    
    # Euclidean distance statistics
    mean_euclidean = np.mean(euclidean_distances) if len(euclidean_distances) > 0 else 0.0
    median_euclidean = np.median(euclidean_distances) if len(euclidean_distances) > 0 else 0.0
    min_euclidean = np.min(euclidean_distances) if len(euclidean_distances) > 0 else 0.0
    max_euclidean = np.max(euclidean_distances) if len(euclidean_distances) > 0 else 0.0
    std_euclidean = np.std(euclidean_distances) if len(euclidean_distances) > 0 else 0.0

    # Method: For each model point, find the nearest GT point using Euclidean distance
    # A model point is considered "correct" if its distance to nearest GT <= threshold
    # Then calculate Precision, Recall, and F1 based on this matching
    
    # For each model point, find minimum Euclidean distance to any GT point
    correct_model_points = 0  # Model points within threshold
    matched_gt_indices = set()  # GT points that have been matched
    
    for mx, my, mz in model_points:
        # Find nearest GT point
        min_dist = float('inf')
        nearest_gt_idx = -1
        
        for gi, (gx, gy, gz) in enumerate(gt_points):
            dist = calculate_distance(mx, my, mz, gx, gy, gz)
            if dist < min_dist:
                min_dist = dist
                nearest_gt_idx = gi
        
        # If distance to nearest GT <= threshold, this model point is correct
        if min_dist <= tol:
            correct_model_points += 1
            if nearest_gt_idx >= 0:
                matched_gt_indices.add(nearest_gt_idx)
    
    matches = correct_model_points  # Number of correct model points
    matched_gt_count = len(matched_gt_indices)  # Number of unique GT points matched

    # Precision: Of all model points, how many are correct (within threshold)?
    precision = correct_model_points / model_count if model_count > 0 else 0.0
    
    # Recall: Of all GT points, how many were found by the model (matched)?
    recall = matched_gt_count / gt_count if gt_count > 0 else 0.0
    
    # F1 Score: Harmonic mean of Precision and Recall
    f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    union = model_count + gt_count - matches
    accuracy_iou = matches / union if union > 0 else 0.0

    return {
        'matches': matches,
        'precision': precision,
        'recall': recall,
        'f1_score': f1_score,
        'model_count': model_count,
        'gt_count': gt_count,
        'accuracy_iou': accuracy_iou,
        'mean_euclidean_distance': mean_euclidean,
        'median_euclidean_distance': median_euclidean,
        'min_euclidean_distance': min_euclidean,
        'max_euclidean_distance': max_euclidean,
        'std_euclidean_distance': std_euclidean,
    }


# ============================================================================
# 7. Main Execution Loop
# ============================================================================
if __name__ == "__main__":
    print("="*70)
    print("SCALE EXPERIMENT: Complete Pipeline (Euclidean Distance Based)")
    print("="*70)
    print("\nThis will:")
    print("  1. Create ground truth (synthetic)")
    print("  2. Generate subgraphs")
    print("  3. Train model")
    print("  4. Extract coordinates from model and GT")
    print("  5. Compute accuracy with Euclidean distance metrics")
    print("="*70)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nDevice: {device}\n")

    results = []

    for scale_name, x_scale, y_scale, z_scale in SCALES:
        print(f"\n{'#'*70}")
        print(f"Processing Scale: {scale_name} (X={x_scale}x, Y={y_scale}x, Z={z_scale}x)")
        print(f"{'#'*70}\n")

        # 1. Create ground truth data
        data, gt_filepath = create_groundtruth(scale_name, x_scale, y_scale, z_scale, BASE_DIR)

        # 2. Create subgraphs
        graphs, subgraphs_filepath = create_subgraphs(data, scale_name, BASE_DIR)

        # 3. Filter graphs
        min_nodes = 100
        filtered_graphs = [g for g in graphs if g.x.shape[0] >= min_nodes]
        print(f"\n  Filtered to {len(filtered_graphs)} graphs (min {min_nodes} nodes)")

        if len(filtered_graphs) == 0:
            print(f"  ⚠ No graphs after filtering! Skipping {scale_name}...")
            continue

        # 4. Attach GT scores to graphs for supervised training
        print(f"\n  Attaching GT scores to graphs...")
        filtered_graphs = attach_gt_scores_to_graphs(data, filtered_graphs)

        # 5. Train model
        print(f"\n  Training model for {scale_name}...")
        in_channels = max([g.x.shape[1] for g in filtered_graphs])
        edge_dim = 1 if any(hasattr(g, 'edge_attr') and g.edge_attr is not None for g in filtered_graphs) else None

        model = ContrastiveGAT(
            in_channels=in_channels,
            hidden_channels=32,
            projection_dim=16,
            num_heads=4,
            dropout=0.1,
            edge_dim=edge_dim
        ).to(device)

        subsampled_graphs = filtered_graphs[::2] if len(filtered_graphs) > 20000 else filtered_graphs
        model, losses = train_contrastive_model(
            model=model,
            graphs=subsampled_graphs,
            device=device,
            epochs=1,  # Increased from 1 to 20 for better learning
            batch_size=32,
            lr=0.1,  # Reduced from 0.01 to 0.001 for more stable training
            gradient_accumulation_steps=4
        )

        # Save model
        model_filepath = os.path.join(BASE_DIR, f'model_{scale_name}.pt')
        model_info = {
            'model_state_dict': model.state_dict(),
            'in_channels': in_channels,
            'hidden_channels': 32,
            'projection_dim': 16,
            'num_heads': 4,
            'dropout': 0.1,
            'edge_dim': edge_dim,
            'losses': losses
        }
        torch.save(model_info, model_filepath)
        print(f"  ✓ Saved model to: {model_filepath}")

        # 6. Run GAT model to extract top positions (reconstruct and find top positions)
        print(f"\n  Running GAT model to extract top positions...")
        model_coords = find_max_interaction_positions(model, filtered_graphs, device, top_k=TOP_K)

        # Save model_coords as pickle (for internal use)
        model_coords_filepath = os.path.join(BASE_DIR, f'model_coords_{scale_name}.pkl')
        with open(model_coords_filepath, 'wb') as f:
            pickle.dump(model_coords, f)
        print(f"  ✓ Saved model coordinates to: {model_coords_filepath}")

        # Save top_positions_result as pickle
        top_positions_pkl_filepath = os.path.join(BASE_DIR, f'top_positions_result_{scale_name}.pkl')
        with open(top_positions_pkl_filepath, 'wb') as f:
            pickle.dump(model_coords, f)
        print(f"  ✓ Saved top positions result (pkl) to: {top_positions_pkl_filepath}")

        # Save top_positions_result as numpy array (extract x, y, z coordinates)
        top_positions_array = np.array([[c['x'], c['y'], c['z']] for c in model_coords], dtype=np.int32)
        top_positions_npy_filepath = os.path.join(BASE_DIR, f'top_positions_result_{scale_name}.npy')
        np.save(top_positions_npy_filepath, top_positions_array)
        print(f"  ✓ Saved top positions result (npy) to: {top_positions_npy_filepath}")

        # 7. Extract ground-truth coordinates from data (Option A)
        # IMPORTANT: Use filtered_graphs (same as model training) for fair comparison
        print(f"\n  Extracting coordinates from ground truth (data, using filtered_graphs)...")
        gt_coords = extract_gt_coordinates_from_data(data, filtered_graphs, top_k=TOP_K)
        gt_coords_filepath = os.path.join(BASE_DIR, f'gt_coords{scale_name}.pkl')
        with open(gt_coords_filepath, 'wb') as f:
            pickle.dump(gt_coords, f)
        print(f"  ✓ Saved GT coordinates to: {gt_coords_filepath}")

        # 8. Compute accuracy for multiple thresholds
        print(f"\n  Computing accuracy for thresholds: {EUCLIDEAN_THRESHOLDS}...")
        
        for threshold in EUCLIDEAN_THRESHOLDS:
            accuracy = compute_accuracy(model_coords, gt_coords, tol=threshold)
            results.append({
                'Scale Name': scale_name,
                'Threshold': threshold,
                'F1 Score': f"{accuracy['f1_score']:.4f}",
                'Accuracy (IoU)': f"{accuracy['accuracy_iou']:.4f}",
            })
            print(
                f"  ✓ Threshold={threshold}: "
                f"F1={accuracy['f1_score']:.4f}, "
                f"Accuracy={accuracy['accuracy_iou']:.4f}"
            )

    # 8. Create final results table
    print(f"\n{'='*70}")
    print("FINAL RESULTS TABLE")
    print(f"{'='*70}\n")

    results_df = pd.DataFrame(results)
    
    # Create pivot tables for better visualization
    print("F1 Score by Scale and Threshold:")
    print("-" * 70)
    pivot_f1 = results_df.pivot(index='Scale Name', columns='Threshold', values='F1 Score')
    print(pivot_f1.to_string())
    print("\n")
    
    print("Accuracy (IoU) by Scale and Threshold:")
    print("-" * 70)
    pivot_accuracy = results_df.pivot(index='Scale Name', columns='Threshold', values='Accuracy (IoU)')
    print(pivot_accuracy.to_string())
    print("\n")
    
    # Save results table
    results_filepath = os.path.join(BASE_DIR, 'accuracy_results.csv')
    results_df.to_csv(results_filepath, index=False)
    
    # Save pivot tables
    pivot_f1.to_csv(os.path.join(BASE_DIR, 'f1_score_by_threshold.csv'))
    pivot_accuracy.to_csv(os.path.join(BASE_DIR, 'accuracy_by_threshold.csv'))
    
    print(f"✓ Results saved to: {results_filepath}")
    print(f"✓ Pivot tables saved to: {BASE_DIR}")

    print(f"\n{'='*70}")
    print("EXPERIMENT COMPLETE!")
    print(f"{'='*70}\n")


SCALE EXPERIMENT: Complete Pipeline (Euclidean Distance Based)

This will:
  1. Create ground truth (synthetic)
  2. Generate subgraphs
  3. Train model
  4. Extract coordinates from model and GT
  5. Compute accuracy with Euclidean distance metrics

Device: cpu


######################################################################
Processing Scale: original (X=1x, Y=1x, Z=1x)
######################################################################


Creating Ground Truth: original (X=1x, Y=1x, Z=1x)
  Dimensions: X=172, Y=87, Z=12
✓ Saved ground truth to: D:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale\groundtruth_original.npy
  Shape: (4, 11, 12, 87, 172), Size: 7.53 MB

Creating Subgraphs: original
  Using z=5, Dimensions: 87x172
  Pre-computing intensity matrix...
  Total graph centers: 630
  Creating subgraphs...


Processing: 100%|██████████| 630/630 [00:15<00:00, 39.40it/s] 


✓ Saved 566 subgraphs to: D:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale\Subgraph_original.pt
  File size: 129.79 MB

  Filtered to 286 graphs (min 100 nodes)

  Attaching GT scores to graphs...

  Training model for original...

Training Contrastive GAT Model
  Epochs: 1, Batch size: 32, LR: 0.1
  Total graphs: 286, Target channels: 4
Epoch 1/1 - Loss: 4.543684

✓ Training complete!
  ✓ Saved model to: D:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale\model_original.pt

  Running GAT model to extract top positions...


Processing graphs (model): 100%|██████████| 286/286 [00:04<00:00, 68.75it/s]


  ✓ Saved model coordinates to: D:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale\model_coords_original.pkl
  ✓ Saved top positions result (pkl) to: D:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale\top_positions_result_original.pkl
  ✓ Saved top positions result (npy) to: D:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale\top_positions_result_original.npy

  Extracting coordinates from ground truth (data, using filtered_graphs)...

Computing ground-truth coordinates from data (Option A: ANY channel nonzero)...


...

✓ Saved 2244 subgraphs to: D:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale\Subgraph_3x2y1z.pt
  File size: 773.95 MB

  Filtered to 1449 graphs (min 100 nodes)

  Attaching GT scores to graphs...

  Training model for 3x2y1z...

Training Contrastive GAT Model
  Epochs: 1, Batch size: 32, LR: 0.1
  Total graphs: 1449, Target channels: 3
Epoch 1/1 - Loss: 3.668923

✓ Training complete!
  ✓ Saved model to: D:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale\model_3x2y1z.pt

  Running GAT model to extract top positions...


Processing graphs (model): 100%|██████████| 1449/1449 [01:51<00:00, 12.96it/s]


  ✓ Saved model coordinates to: D:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale\model_coords_3x2y1z.pkl
  ✓ Saved top positions result (pkl) to: D:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale\top_positions_result_3x2y1z.pkl
  ✓ Saved top positions result (npy) to: D:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale\top_positions_result_3x2y1z.npy

  Extracting coordinates from ground truth (data, using filtered_graphs)...

Computing ground-truth coordinates from data (Option A: ANY channel nonzero)...


Processing graphs (GT): 100%|██████████| 1449/1449 [00:00<00:00, 6692.37it/s]


  ✓ Saved GT coordinates to: D:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale\gt_coords3x2y1z.pkl

  Computing accuracy for thresholds: [5, 10, 15]...
  ✓ Threshold=5: F1=0.8950, Accuracy=1.0000
  ✓ Threshold=10: F1=0.8950, Accuracy=1.0000
  ✓ Threshold=15: F1=0.8950, Accuracy=1.0000

######################################################################
Processing Scale: 3x1y2z (X=3x, Y=1x, Z=2x)
######################################################################


Creating Ground Truth: 3x1y2z (X=3x, Y=1x, Z=2x)
  Dimensions: X=516, Y=87, Z=24


======================================================================
FINAL RESULTS TABLE
======================================================================

Scale Name    Scale                  GT File  Matches  Model Points  GT Points Precision Recall F1 Score Accuracy (IoU) Mean Euclidean Distance Median Euclidean Distance Min Euclidean Distance Max Euclidean Distance Std Euclidean Distance
  original 1x,1x,1x groundtruth_original.npy      100           100        100    1.0000 0.8600   0.9247         1.0000                    0.70                      0.00                   0.00                   5.00                   1.73
    2x1y1z 2x,1x,1x   groundtruth_2x1y1z.npy      100           100        100    1.0000 0.8600   0.9247         1.0000                    0.70                      0.00                   0.00                   5.00                   1.73
    1x2y1z 1x,2x,1x   groundtruth_1x2y1z.npy      100           100        100    1.0000 0.8500   0.9189         1.0000                    0.75                      0.00                   0.00                   5.00                   1.79
    1x1y2z 1x,1x,2x   groundtruth_1x1y2z.npy      100           100        100    1.0000 0.9200   0.9583         1.0000                    0.40                      0.00                   0.00                   5.00                   1.36
    2x2y1z 2x,2x,1x   groundtruth_2x2y1z.npy      100           100        100    1.0000 0.8600   0.9247         1.0000                    0.70                      0.00                   0.00                   5.00                   1.73
    1x2y2z 1x,2x,2x   groundtruth_1x2y2z.npy      100           100        100    1.0000 0.8600   0.9247         1.0000                    0.75                      0.00                   0.00                  10.00                   1.92
    2x2y2z 2x,2x,2x   groundtruth_2x2y2z.npy      100           100        100    1.0000 0.8600   0.9247         1.0000                    0.70                      0.00                   0.00                   5.00                   1.73
    3x2y1z 3x,2x,1x   groundtruth_3x2y1z.npy      100           100        100    1.0000 0.7000   0.8235         1.0000                    1.50                      0.00                   0.00                   5.00                   2.29
    3x1y2z 3x,1x,2x   groundtruth_3x1y2z.npy      100           100        100    1.0000 0.8700   0.9305         1.0000                    0.65                      0.00                   0.00                   5.00                   1.68
    3x2y2z 3x,2x,2x   groundtruth_3x2y2z.npy      100           100        100    1.0000 0.8200   0.9011         1.0000                    0.90                      0.00                   0.00                   5.00                   1.92

✓ Results saved to: D:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale/accuracy_results.csv

======================================================================
EXPERIMENT COMPLETE!
======================================================================

#synthetic texture euclidean distance 

In [ ]:
"""
Complete texture Experiment Script
Creates ground truth, trains model, extracts coordinates, and computes accuracy
(radius-aware IoU)
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool, global_add_pool
from torch_geometric.data import Data, Batch
import os
import time
import pandas as pd
from tqdm import tqdm
import pickle
import random
import matplotlib.pyplot as plt

# ============================================================================
# Configuration
# ============================================================================

# Portable BASE_DIR: works on Windows, Linux, and Colab
BASE_DIR = os.path.join(os.getcwd(), "Groundtruth", "Texture")
os.makedirs(BASE_DIR, exist_ok=True)

# Original dimensions
ORIGINAL_X = 172
ORIGINAL_Y = 87
ORIGINAL_Z = 12

# Texture types
TEXTURES = ['texture1', 'texture2', 'texture3', 'texture4', 'texture5', 'texture6', 'texture7']
TOP_K = 100
Z_FIXED = 5
MAX_RADIUS = 10            # Used for graphs, GT scoring, and tolerance in IoU
STEP_SIZE = 5

# ============================================================================
# Helper Functions
# ============================================================================

def calculate_distance(x1, y1, z1, x2, y2, z2):
    """Calculate 3D Euclidean distance between two points"""
    return np.sqrt((x2 - x1)**2 + (y2 - y1)**2 + (z2 - z1)**2)


# ============================================================================
# 1. Create Ground Truth with Different Textures and Scales
# ============================================================================

def create_groundtruth(texture_type, output_dir):
    """Create ground truth data with specified texture type"""
    print(f"\n{'='*70}")
    print(f"Creating Ground Truth: {texture_type}")
    print(f"{'='*70}")

    num_channels = 4
    value_range = (0, 10)

    # Use original dimensions (no scaling)
    x_dim = ORIGINAL_X
    y_dim = ORIGINAL_Y
    z_dim = ORIGINAL_Z
    num_values = value_range[1] - value_range[0] + 1

    print(f"  Dimensions: X={x_dim}, Y={y_dim}, Z={z_dim}")

    # Create 5D array: (C, V, Z, Y, X)
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)

    # Generate patterns based on texture type (scaled)
    for z in range(z_dim):
        if texture_type == 'texture1':
            # Texture 1: Circular/Radial patterns (concentric circles)
            center_x, center_y = x_dim // 2, y_dim // 2
            for i in range(3):
                radius = int(15 + 20 * i)
                for y in range(y_dim):
                    for x in range(x_dim):
                        dist = np.sqrt((x - center_x)**2 + (y - center_y)**2)
                        if abs(dist - radius) <= 2:
                            random_value = np.random.randint(6, 11)
                            data[0, random_value, z, y, x] = 1

            # Channel 1: Wave patterns (sinusoidal waves)
            for i in range(3):
                wave_center_y = int(20 + 20 * i)
                amplitude = int(15)
                frequency = 0.1
                for x in range(x_dim):
                    wave_y = int(wave_center_y + amplitude * np.sin(frequency * x))
                    if 0 <= wave_y < y_dim:
                        for dy in range(-2, 3):
                            y = wave_y + dy
                            if 0 <= y < y_dim:
                                random_value = np.random.randint(6, 11)
                                data[1, random_value, z, y, x] = 1

            # Channel 2: Spot/Blob patterns (circular spots)
            spot_centers = [
                (int(40), int(30)),
                (int(100), int(50)),
                (int(130), int(70))
            ]
            for center_x_spot, center_y_spot in spot_centers:
                spot_radius = int(8)
                for y in range(y_dim):
                    for x in range(x_dim):
                        dist = np.sqrt((x - center_x_spot)**2 + (y - center_y_spot)**2)
                        if dist <= spot_radius:
                            intensity = int(10 - (dist / spot_radius) * 4)
                            intensity = max(6, min(10, intensity))
                            random_value = np.random.randint(intensity, 11)
                            data[2, random_value, z, y, x] = 1

            # Channel 3: Checkerboard/Grid patterns
            grid_size = int(12)
            for y in range(y_dim):
                for x in range(x_dim):
                    if (x // grid_size + y // grid_size) % 2 == 0:
                        if (x % grid_size < 3) or (y % grid_size < 3):
                            random_value = np.random.randint(6, 11)
                            data[3, random_value, z, y, x] = 1

        elif texture_type == 'texture2':
            # Texture 2: Linear patterns (vertical, horizontal, diagonal)
            for i in range(3):
                # Channel 0: Vertical lines (scaled)
                x_pos = int(40 + 40 * i)
                if 0 <= x_pos < x_dim:
                    for y in range(y_dim):
                        for dx in range(-2, 3):
                            x = x_pos + dx
                            if 0 <= x < x_dim:
                                random_value = np.random.randint(6, 11)
                                data[0, random_value, z, y, x] = 1

                # Channel 1: Horizontal lines (scaled)
                y_pos = int(20 + 20 * i)
                if 0 <= y_pos < y_dim:
                    for x in range(x_dim):
                        for dy in range(-2, 3):
                            y = y_pos + dy
                            if 0 <= y < y_dim:
                                random_value = np.random.randint(6, 11)
                                data[1, random_value, z, y, x] = 1

            # Channel 2: Diagonal lines with slope 1 (scaled)
            strip_width = int(3)
            c_values = [-int(60), 0]
            for c in c_values:
                for y in range(y_dim):
                    for x in range(x_dim):
                        if abs(y * (y_dim/ORIGINAL_Y) - x * (x_dim/ORIGINAL_X) - c) <= strip_width:
                            random_value = np.random.randint(6, 11)
                            data[2, random_value, z, y, x] = 1

            # Channel 3: Diagonal lines with slope -1 (scaled)
            d_values = [int(60), int(120)]
            for d in d_values:
                for y in range(y_dim):
                    for x in range(x_dim):
                        if abs(y * (y_dim/ORIGINAL_Y) + x * (x_dim/ORIGINAL_X) - d) <= strip_width:
                            random_value = np.random.randint(6, 11)
                            data[3, random_value, z, y, x] = 1

        elif texture_type == 'texture3':
            # Texture 3: Spiral and radial patterns
            center_x, center_y = x_dim // 2, y_dim // 2
            # Channel 0: Spiral patterns
            for i in range(3):
                spiral_turns = 2 + i
                for angle_idx in range(0, 360 * spiral_turns, 5):
                    angle = np.radians(angle_idx)
                    radius = int((10 + 15 * i) * (angle_idx / (360 * spiral_turns)))
                    x = int(center_x + radius * np.cos(angle))
                    y = int(center_y + radius * np.sin(angle))
                    if 0 <= x < x_dim and 0 <= y < y_dim:
                        random_value = np.random.randint(6, 11)
                        data[0, random_value, z, y, x] = 1

            # Channel 1: Star patterns (radial lines)
            for i in range(8):
                angle = np.radians(i * 45)
                for r in range(0, min(x_dim, y_dim) // 2, 2):
                    x = int(center_x + r * np.cos(angle))
                    y = int(center_y + r * np.sin(angle))
                    if 0 <= x < x_dim and 0 <= y < y_dim:
                        random_value = np.random.randint(6, 11)
                        data[1, random_value, z, y, x] = 1

            # Channel 2: Concentric squares
            for i in range(3):
                square_size = int(15 + 20 * i)
                for offset in range(-square_size, square_size + 1):
                    for coord in [center_x + offset, center_x - offset]:
                        if 0 <= coord < x_dim:
                            for y in range(max(0, center_y - square_size), min(y_dim, center_y + square_size + 1)):
                                if abs(y - center_y) == square_size or abs(coord - center_x) == square_size:
                                    random_value = np.random.randint(6, 11)
                                    data[2, random_value, z, y, coord] = 1

            # Channel 3: Random clusters
            for i in range(5):
                cluster_x = np.random.randint(0, x_dim)
                cluster_y = np.random.randint(0, y_dim)
                cluster_radius = int(10)
                for y in range(max(0, cluster_y - cluster_radius), min(y_dim, cluster_y + cluster_radius + 1)):
                    for x in range(max(0, cluster_x - cluster_radius), min(x_dim, cluster_x + cluster_radius + 1)):
                        dist = np.sqrt((x - cluster_x)**2 + (y - cluster_y)**2)
                        if dist <= cluster_radius:
                            if np.random.random() < 0.3:  # Sparse clusters
                                random_value = np.random.randint(6, 11)
                                data[3, random_value, z, y, x] = 1

        elif texture_type == 'texture4':
            # Texture 4: Grid and wave patterns
            center_x, center_y = x_dim // 2, y_dim // 2

            # Channel 0: Regular grid pattern
            grid_spacing = 20
            for y in range(0, y_dim, grid_spacing):
                for x in range(x_dim):
                    random_value = np.random.randint(6, 11)
                    data[0, random_value, z, y, x] = 1
            for x in range(0, x_dim, grid_spacing):
                for y in range(y_dim):
                    random_value = np.random.randint(6, 11)
                    data[0, random_value, z, y, x] = 1

            # Channel 1: Concentric circles with varying density
            for i in range(5):
                radius = int(10 + 15 * i)
                for y in range(y_dim):
                    for x in range(x_dim):
                        dist = np.sqrt((x - center_x)**2 + (y - center_y)**2)
                        if abs(dist - radius) <= 1:
                            random_value = np.random.randint(6, 11)
                            data[1, random_value, z, y, x] = 1

            # Channel 2: Horizontal waves
            for i in range(3):
                wave_center_y = int(20 + 25 * i)
                amplitude = int(20)
                frequency = 0.05
                for x in range(x_dim):
                    wave_y = int(wave_center_y + amplitude * np.sin(frequency * x))
                    if 0 <= wave_y < y_dim:
                        for dy in range(-3, 4):
                            y = wave_y + dy
                            if 0 <= y < y_dim:
                                random_value = np.random.randint(6, 11)
                                data[2, random_value, z, y, x] = 1

            # Channel 3: Diagonal stripes
            stripe_width = 5
            for i in range(4):
                offset = int(-80 + 40 * i)
                for y in range(y_dim):
                    for x in range(x_dim):
                        if abs(y - x - offset) <= stripe_width:
                            random_value = np.random.randint(6, 11)
                            data[3, random_value, z, y, x] = 1

        elif texture_type == 'texture5':
            # Texture 5: Cross patterns, zigzag, and diamond shapes
            center_x, center_y = x_dim // 2, y_dim // 2

            # Channel 0: Cross patterns (plus signs)
            for i in range(4):
                cross_x = int(30 + 35 * i)
                cross_y = int(20 + 20 * i)
                cross_size = int(12)
                if 0 <= cross_x < x_dim and 0 <= cross_y < y_dim:
                    # Horizontal line
                    for dx in range(-cross_size, cross_size + 1):
                        x = cross_x + dx
                        if 0 <= x < x_dim:
                            random_value = np.random.randint(6, 11)
                            data[0, random_value, z, cross_y, x] = 1
                    # Vertical line
                    for dy in range(-cross_size, cross_size + 1):
                        y = cross_y + dy
                        if 0 <= y < y_dim:
                            random_value = np.random.randint(6, 11)
                            data[0, random_value, z, y, cross_x] = 1

            # Channel 1: Zigzag patterns
            for i in range(3):
                zigzag_start_y = int(15 + 25 * i)
                zigzag_width = int(8)
                zigzag_period = int(25)
                for x in range(x_dim):
                    zigzag_y = int(zigzag_start_y + zigzag_width * np.sin(2 * np.pi * x / zigzag_period))
                    if 0 <= zigzag_y < y_dim:
                        for dy in range(-2, 3):
                            y = zigzag_y + dy
                            if 0 <= y < y_dim:
                                random_value = np.random.randint(6, 11)
                                data[1, random_value, z, y, x] = 1

            # Channel 2: Diamond shapes
            for i in range(3):
                diamond_x = int(50 + 40 * i)
                diamond_y = int(30 + 25 * i)
                diamond_size = int(10)
                if 0 <= diamond_x < x_dim and 0 <= diamond_y < y_dim:
                    for y in range(max(0, diamond_y - diamond_size), min(y_dim, diamond_y + diamond_size + 1)):
                        for x in range(max(0, diamond_x - diamond_size), min(x_dim, diamond_x + diamond_size + 1)):
                            dx = abs(x - diamond_x)
                            dy = abs(y - diamond_y)
                            if dx + dy <= diamond_size:
                                random_value = np.random.randint(6, 11)
                                data[2, random_value, z, y, x] = 1

            # Channel 3: Radial spokes (like a wheel)
            num_spokes = 12
            for i in range(num_spokes):
                angle = np.radians(i * 360 / num_spokes)
                for r in range(5, min(x_dim, y_dim) // 2, 2):
                    x = int(center_x + r * np.cos(angle))
                    y = int(center_y + r * np.sin(angle))
                    if 0 <= x < x_dim and 0 <= y < y_dim:
                        random_value = np.random.randint(6, 11)
                        data[3, random_value, z, y, x] = 1

        elif texture_type == 'texture6':
            # Texture 6: Hexagonal patterns, curved lines, elliptical shapes, wavy diagonals
            center_x, center_y = x_dim // 2, y_dim // 2

            # Channel 0: Hexagonal patterns
            hex_size = 15
            for i in range(3):
                hex_center_x = int(40 + 50 * i)
                hex_center_y = int(30 + 30 * i)
                if 0 <= hex_center_x < x_dim and 0 <= hex_center_y < y_dim:
                    for angle_deg in range(0, 360, 60):
                        angle = np.radians(angle_deg)
                        for r in range(hex_size):
                            x = int(hex_center_x + r * np.cos(angle))
                            y = int(hex_center_y + r * np.sin(angle))
                            if 0 <= x < x_dim and 0 <= y < y_dim:
                                random_value = np.random.randint(6, 11)
                                data[0, random_value, z, y, x] = 1

            # Channel 1: Curved lines/arcs
            for i in range(4):
                arc_center_x = int(30 + 35 * i)
                arc_center_y = int(20 + 20 * i)
                arc_radius = int(25)
                for angle_deg in range(0, 180, 3):
                    angle = np.radians(angle_deg)
                    x = int(arc_center_x + arc_radius * np.cos(angle))
                    y = int(arc_center_y + arc_radius * np.sin(angle))
                    if 0 <= x < x_dim and 0 <= y < y_dim:
                        for dx in range(-1, 2):
                            for dy in range(-1, 2):
                                nx, ny = x + dx, y + dy
                                if 0 <= nx < x_dim and 0 <= ny < y_dim:
                                    random_value = np.random.randint(6, 11)
                                    data[1, random_value, z, ny, nx] = 1

            # Channel 2: Elliptical shapes
            for i in range(3):
                ellipse_x = int(50 + 40 * i)
                ellipse_y = int(35 + 25 * i)
                ellipse_a = int(20)  # semi-major axis
                ellipse_b = int(12)  # semi-minor axis
                if 0 <= ellipse_x < x_dim and 0 <= ellipse_y < y_dim:
                    for y in range(max(0, ellipse_y - ellipse_b), min(y_dim, ellipse_y + ellipse_b + 1)):
                        for x in range(max(0, ellipse_x - ellipse_a), min(x_dim, ellipse_x + ellipse_a + 1)):
                            dx = (x - ellipse_x) / ellipse_a
                            dy = (y - ellipse_y) / ellipse_b
                            if dx*dx + dy*dy <= 1.0:
                                random_value = np.random.randint(6, 11)
                                data[2, random_value, z, y, x] = 1

            # Channel 3: Wavy diagonal lines
            for i in range(3):
                wave_amplitude = int(8)
                wave_frequency = 0.1
                base_offset = int(-60 + 60 * i)
                for x in range(x_dim):
                    wave_offset = int(wave_amplitude * np.sin(wave_frequency * x))
                    y = x + base_offset + wave_offset
                    if 0 <= y < y_dim:
                        for dy in range(-2, 3):
                            ny = y + dy
                            if 0 <= ny < y_dim:
                                random_value = np.random.randint(6, 11)
                                data[3, random_value, z, ny, x] = 1

        elif texture_type == 'texture7':
            # Texture 7: Lattice patterns, concentric hexagons, parallel curves, connected dots
            center_x, center_y = x_dim // 2, y_dim // 2

            # Channel 0: Lattice patterns (interlaced grid)
            lattice_spacing = 18
            for y in range(0, y_dim, lattice_spacing):
                for x in range(x_dim):
                    if (x // lattice_spacing) % 2 == 0:
                        random_value = np.random.randint(6, 11)
                        data[0, random_value, z, y, x] = 1
            for x in range(0, x_dim, lattice_spacing):
                for y in range(y_dim):
                    if (y // lattice_spacing) % 2 == 0:
                        random_value = np.random.randint(6, 11)
                        data[0, random_value, z, y, x] = 1

            # Channel 1: Concentric hexagons
            for i in range(4):
                hex_radius = int(8 + 12 * i)
                for angle_deg in range(0, 360, 6):
                    angle = np.radians(angle_deg)
                    x = int(center_x + hex_radius * np.cos(angle))
                    y = int(center_y + hex_radius * np.sin(angle))
                    if 0 <= x < x_dim and 0 <= y < y_dim:
                        # Create hexagon by connecting points
                        next_angle = np.radians(angle_deg + 6)
                        next_x = int(center_x + hex_radius * np.cos(next_angle))
                        next_y = int(center_y + hex_radius * np.sin(next_angle))
                        # Draw line between points
                        steps = max(abs(next_x - x), abs(next_y - y))
                        if steps > 0:
                            for s in range(steps + 1):
                                line_x = int(x + (next_x - x) * s / steps)
                                line_y = int(y + (next_y - y) * s / steps)
                                if 0 <= line_x < x_dim and 0 <= line_y < y_dim:
                                    random_value = np.random.randint(6, 11)
                                    data[1, random_value, z, line_y, line_x] = 1

            # Channel 2: Parallel curves (sine waves with different phases)
            for i in range(3):
                curve_center_y = int(20 + 25 * i)
                curve_amplitude = int(15)
                curve_frequency = 0.08
                phase = i * np.pi / 3
                for x in range(x_dim):
                    curve_y = int(curve_center_y + curve_amplitude * np.sin(curve_frequency * x + phase))
                    if 0 <= curve_y < y_dim:
                        for dy in range(-2, 3):
                            y = curve_y + dy
                            if 0 <= y < y_dim:
                                random_value = np.random.randint(6, 11)
                                data[2, random_value, z, y, x] = 1

            # Channel 3: Connected scattered dots
            dot_centers = [
                (int(30), int(20)), (int(60), int(30)), (int(90), int(25)),
                (int(120), int(40)), (int(150), int(35)), (int(80), int(50)),
                (int(50), int(60)), (int(100), int(65))
            ]
            for center_x_dot, center_y_dot in dot_centers:
                if 0 <= center_x_dot < x_dim and 0 <= center_y_dot < y_dim:
                    # Draw dot
                    dot_radius = 3
                    for y in range(max(0, center_y_dot - dot_radius), min(y_dim, center_y_dot + dot_radius + 1)):
                        for x in range(max(0, center_x_dot - dot_radius), min(x_dim, center_x_dot + dot_radius + 1)):
                            if np.sqrt((x - center_x_dot)**2 + (y - center_y_dot)**2) <= dot_radius:
                                random_value = np.random.randint(6, 11)
                                data[3, random_value, z, y, x] = 1
            
            # Connect nearby dots
            for i, (x1, y1) in enumerate(dot_centers):
                for j, (x2, y2) in enumerate(dot_centers[i+1:], i+1):
                    dist = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
                    if dist < 50:  # Connect if close enough
                        steps = int(dist)
                        if steps > 0:
                            for s in range(steps + 1):
                                line_x = int(x1 + (x2 - x1) * s / steps)
                                line_y = int(y1 + (y2 - y1) * s / steps)
                                if 0 <= line_x < x_dim and 0 <= line_y < y_dim:
                                    random_value = np.random.randint(6, 11)
                                    data[3, random_value, z, line_y, line_x] = 1
                                    # Save ground truth
    filename = f'groundtruth_{texture_type}.npy'
    filepath = os.path.join(output_dir, filename)
    np.save(filepath, data)
    print(f"✓ Saved ground truth to: {filepath}")
    print(f"  Shape: {data.shape}, Size: {data.nbytes / 1024 / 1024:.2f} MB")

    return data, filepath


# ============================================================================
# 1b. Visualize Ground Truth (Colorful Plot)
# ============================================================================

def visualize_groundtruth(data, texture_type, output_dir, z_slice=None):
    """
    Create colorful visualization of ground truth data.
    We project over "value" and "z" to get a 2D image per channel, then blend
    channels into RGB.
    """
    num_channels, num_values, z_dim, y_dim, x_dim = data.shape

    # Predefine an RGB color for each channel (R, G, B)
    channel_colors = np.array([
        [1.0, 0.0, 0.0],  # Channel 0: red
        [0.0, 1.0, 0.0],  # Channel 1: green
        [0.0, 0.0, 1.0],  # Channel 2: blue
        [1.0, 0.0, 1.0],  # Channel 3: magenta
    ], dtype=np.float32)

    # Store the 2D projection per channel
    per_channel_2d = np.zeros((num_channels, y_dim, x_dim), dtype=np.float32)

    for channel in range(num_channels):
        channel_data = data[channel]  # Shape: (value, z, y, x)

        # value indices: 0..(num_values-1)
        value_indices = np.arange(num_values).reshape(-1, 1, 1, 1)  # (value, 1, 1, 1)

        # Weighted sum over "value" axis
        weighted_sum = np.sum(value_indices * channel_data, axis=0)  # (z, y, x)
        total_sum = np.sum(channel_data, axis=0)                     # (z, y, x)

        avg_value_index = np.where(total_sum > 0, weighted_sum / total_sum, 0)  # (z, y, x)

        # If a specific z_slice is requested, use that; otherwise average over z
        if z_slice is not None and 0 <= z_slice < z_dim:
            avg_z = avg_value_index[z_slice]  # (y, x)
        else:
            avg_z = np.mean(avg_value_index, axis=0)  # (y, x)

        per_channel_2d[channel] = avg_z

    # Build an RGB image from all channels
    combined_rgb = np.zeros((y_dim, x_dim, 3), dtype=np.float32)

    for channel in range(num_channels):
        img = per_channel_2d[channel]

        # Skip empty channels
        if np.all(img == 0):
            continue

        # Normalize this channel's image to [0, 1] for visualization
        img_min = img.min()
        img_max = img.max()
        norm = (img - img_min) / (img_max - img_min + 1e-6)

        # Add this channel's contribution in its own color
        color = channel_colors[channel]  # (3,)
        combined_rgb += norm[..., None] * color  # broadcast to (y, x, 3)

    # Clip to valid [0, 1] RGB range
    combined_rgb = np.clip(combined_rgb, 0.0, 1.0)

    # Plot combined RGB image
    plt.figure(figsize=(12, 8))
    plt.imshow(
        combined_rgb,
        origin='lower',
        extent=[0, x_dim - 1, 0, y_dim - 1],
        aspect='auto',
        interpolation='bilinear'
    )
    plt.xlabel('X')
    plt.ylabel('Y')
    title_extra = f"(z={z_slice})" if z_slice is not None else "avg over z"
    plt.title(f'Combined Channels with Different Colors - {texture_type} {title_extra}')
    plt.xlim(0, x_dim - 1)
    plt.ylim(0, y_dim - 1)
    plt.tight_layout()

    # Save figure
    fig_filename = f'visualization_{texture_type}.png'
    fig_filepath = os.path.join(output_dir, fig_filename)
    plt.savefig(fig_filepath, dpi=150, bbox_inches='tight')
    print(f"✓ Saved visualization to: {fig_filepath}")
    plt.close()

    print(f"\n✓ Combined channels visualization (RGB) for {texture_type}")
    print(f"  RGB image shape: {combined_rgb.shape}")
    print(f"  Per-channel image shape: {per_channel_2d.shape}")

    return fig_filepath


# ============================================================================
# 2. Create Subgraphs from Ground Truth
# ============================================================================

def create_subgraphs(data, texture_type, output_dir):
    """Create subgraphs from ground truth data"""
    print(f"\n{'='*70}")
    print(f"Creating Subgraphs: {texture_type}")
    print(f"{'='*70}")

    num_channels, num_values, z_dim, y_dim, x_dim = data.shape
    z_idx = Z_FIXED if Z_FIXED < z_dim else z_dim // 2

    print(f"  Using z={z_idx}, Dimensions: {y_dim}x{x_dim}")

    # Pre-compute intensity matrix and mask per channel
    print("  Pre-computing intensity matrix...")
    intensity_matrix = np.zeros((y_dim, x_dim, num_channels), dtype=np.float32)
    channel_mask = np.zeros((y_dim, x_dim, num_channels), dtype=bool)

    for channel in range(num_channels):
        # channel_data: (V, Y, X) at fixed z
        channel_data = data[channel, :, z_idx, :, :]
        value_indices = np.argmax(channel_data, axis=0)
        mask = channel_data.sum(axis=0) > 0
        intensity_matrix[:, :, channel] = np.where(mask, value_indices.astype(np.float32), 0.0)
        channel_mask[:, :, channel] = mask

    channel_counts = channel_mask.sum(axis=2)

    # Generate graph centers
    graph_centers = []
    for x in range(0, x_dim, STEP_SIZE):
        for y in range(0, y_dim, STEP_SIZE):
            graph_centers.append((x, y, z_idx))

    print(f"  Total graph centers: {len(graph_centers)}")

    # Create subgraphs
    all_subgraphs = []
    print("  Creating subgraphs...")

    for center_idx, (center_x, center_y, center_z) in enumerate(tqdm(graph_centers, desc="Processing")):
        x_min = max(0, int(center_x - MAX_RADIUS))
        x_max = min(x_dim, int(center_x + MAX_RADIUS) + 1)
        y_min = max(0, int(center_y - MAX_RADIUS))
        y_max = min(y_dim, int(center_y + MAX_RADIUS) + 1)

        nodes = []
        node_positions = []
        node_active_channels = []

        for y in range(y_min, y_max):
            for x in range(x_min, x_max):
                if channel_counts[y, x] > 0:
                    distance = calculate_distance(center_x, center_y, center_z, x, y, z_idx)
                    if distance <= MAX_RADIUS:
                        active_channels = np.where(channel_mask[y, x, :])[0].tolist()
                        intensities = intensity_matrix[y, x, active_channels]
                        nodes.append(intensities)
                        node_positions.append((x, y, z_idx))
                        node_active_channels.append(active_channels)

        if len(nodes) == 0:
            continue

        max_channels = max(len(channels) for channels in node_active_channels)
        padded_nodes = []
        for i, intensities in enumerate(nodes):
            active_ch = node_active_channels[i]
            num_ch = len(active_ch)
            if num_ch < max_channels:
                padded = np.zeros(max_channels, dtype=np.float32)
                padded[:num_ch] = intensities
                padded_nodes.append(padded)
            else:
                padded_nodes.append(intensities)

        node_features = np.array(padded_nodes, dtype=np.float32)
        node_positions_array = np.array(node_positions, dtype=np.int32)
        num_nodes = len(node_features)

        if num_nodes < 2:
            continue

        # Create edges
        node_positions_np = node_positions_array.astype(np.float32)
        if num_nodes < 50000:
            diff = node_positions_np[:, np.newaxis, :] - node_positions_np[np.newaxis, :, :]
            distances_matrix = np.sqrt(np.sum(diff**2, axis=2))
            edge_mask = (distances_matrix <= MAX_RADIUS) & (distances_matrix > 0)
            edge_i, edge_j = np.where(edge_mask)
        else:
            # For large graphs, use iterative approach
            edge_i, edge_j = [], []
            for i in range(num_nodes):
                for j in range(i+1, num_nodes):
                    dist = calculate_distance(
                        node_positions_array[i,0], node_positions_array[i,1], node_positions_array[i,2],
                        node_positions_array[j,0], node_positions_array[j,1], node_positions_array[j,2]
                    )
                    if 0 < dist <= MAX_RADIUS:
                        edge_i.extend([i, j])
                        edge_j.extend([j, i])
            edge_i, edge_j = np.array(edge_i), np.array(edge_j)

        if len(edge_i) == 0:
            continue

        edge_index = torch.tensor([edge_i, edge_j], dtype=torch.long)
        edge_weights_np = np.array([
            calculate_distance(
                node_positions_array[edge_i[k],0], node_positions_array[edge_i[k],1], node_positions_array[edge_i[k],2],
                node_positions_array[edge_j[k],0], node_positions_array[edge_j[k],1], node_positions_array[edge_j[k],2]
            ) for k in range(len(edge_i))
        ], dtype=np.float32)

        node_values = node_features.sum(axis=1)

        # Important for GATConv(edge_dim=1): edge_attr must be shape [E, 1]
        edge_attr = torch.tensor(edge_weights_np, dtype=torch.float32).unsqueeze(1)
        x_tensor = torch.tensor(node_features, dtype=torch.float32)
        node_values_tensor = torch.tensor(node_values, dtype=torch.float32)

        graph = Data(
            x=x_tensor,
            edge_index=edge_index,
            edge_attr=edge_attr,
            center=(center_x, center_y, center_z),
            center_idx=center_idx,
            node_positions=[tuple(pos) for pos in node_positions_array],
            node_values=node_values_tensor,
            max_channels=max_channels
        )

        all_subgraphs.append(graph)

    # Save subgraphs
    filename = f'Subgraph_{texture_type}.pt'
    filepath = os.path.join(output_dir, filename)
    torch.save(all_subgraphs, filepath)
    print(f"✓ Saved {len(all_subgraphs)} subgraphs to: {filepath}")
    print(f"  File size: {os.path.getsize(filepath) / 1024 / 1024:.2f} MB")

    return all_subgraphs, filepath


# ============================================================================
# 3. Model Classes and Functions
# ============================================================================

def prepare_graph_for_batching(graph, target_channels=4):
    """Prepare graph for batching by padding to target_channels"""
    x = graph.x.clone()
    current_channels = x.shape[1]
    if current_channels < target_channels:
        padding = torch.zeros(x.shape[0], target_channels - current_channels, dtype=x.dtype, device=x.device)
        x = torch.cat([x, padding], dim=1)
    elif current_channels > target_channels:
        x = x[:, :target_channels]

    clean_graph = Data(x=x, edge_index=graph.edge_index.clone())
    if hasattr(graph, 'edge_attr') and graph.edge_attr is not None:
        clean_graph.edge_attr = graph.edge_attr.clone()
    if hasattr(graph, 'center'):
        clean_graph._original_center = graph.center

    return clean_graph


def graph_augment(graph, node_mask_ratio=0.1, edge_drop_ratio=0.05, target_channels=4):
    """Augment graph for contrastive learning"""
    aug_graph = prepare_graph_for_batching(graph, target_channels=target_channels)

    # Keep gt_score on augmented graph if present
    if hasattr(graph, 'gt_score'):
        aug_graph.gt_score = graph.gt_score

    num_nodes = aug_graph.x.shape[0]
    num_mask = int(num_nodes * node_mask_ratio)
    if num_mask > 0:
        mask_indices = torch.randperm(num_nodes)[:num_mask]
        aug_graph.x[mask_indices] = 0.0

    if edge_drop_ratio > 0 and aug_graph.edge_index.shape[1] > 0:
        num_edges = aug_graph.edge_index.shape[1]
        num_drop = int(num_edges * edge_drop_ratio)
        if num_drop > 0:
            keep_indices = torch.randperm(num_edges)[:(num_edges - num_drop)]
            aug_graph.edge_index = aug_graph.edge_index[:, keep_indices]
            if hasattr(aug_graph, 'edge_attr') and aug_graph.edge_attr is not None:
                aug_graph.edge_attr = aug_graph.edge_attr[keep_indices]

    return aug_graph


class ContrastiveGAT(nn.Module):
    """Graph Attention Network with Self-Supervised Contrastive Learning"""
    def __init__(self, in_channels=4, hidden_channels=64, projection_dim=32, num_heads=4, dropout=0.1, edge_dim=None):
        super(ContrastiveGAT, self).__init__()
        self.edge_dim = edge_dim
        gat_kwargs = dict(dropout=dropout)
        if self.edge_dim is not None and self.edge_dim > 0:
            gat_kwargs["edge_dim"] = self.edge_dim

        self.gat1 = GATConv(in_channels=in_channels, out_channels=hidden_channels, heads=num_heads, concat=True, **gat_kwargs)
        self.gat2 = GATConv(in_channels=hidden_channels * num_heads, out_channels=hidden_channels, heads=num_heads, concat=True, **gat_kwargs)
        self.gat3 = GATConv(in_channels=hidden_channels * num_heads, out_channels=hidden_channels, heads=1, concat=False, **gat_kwargs)

        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(hidden_channels * num_heads)
        self.norm2 = nn.LayerNorm(hidden_channels * num_heads)
        self.pool_dim = hidden_channels * 3

        self.projection = nn.Sequential(
            nn.Linear(self.pool_dim, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, projection_dim)
        )

        self.interaction_head = nn.Sequential(
            nn.Linear(self.pool_dim, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels // 2, 1)
        )

    def encode(self, x, edge_index, edge_attr=None, batch=None):
        if self.edge_dim is not None and self.edge_dim > 0 and edge_attr is not None:
            ea = edge_attr
        else:
            ea = None

        x = self.gat1(x, edge_index, ea)
        x = self.norm1(x)
        x = F.elu(x)
        x = self.dropout(x)

        x = self.gat2(x, edge_index, ea)
        x = self.norm2(x)
        x = F.elu(x)
        x = self.dropout(x)

        x = self.gat3(x, edge_index, ea)
        x = F.elu(x)

        return x

    def forward(self, x, edge_index, edge_attr=None, batch=None):
        node_emb = self.encode(x, edge_index, edge_attr, batch)

        if batch is None:
            batch = torch.zeros(node_emb.shape[0], dtype=torch.long, device=node_emb.device)

        mean_pool = global_mean_pool(node_emb, batch)
        max_pool = global_max_pool(node_emb, batch)
        sum_pool = global_add_pool(node_emb, batch)

        graph_emb = torch.cat([mean_pool, max_pool, sum_pool], dim=1)

        proj_emb = self.projection(graph_emb)
        proj_emb = F.normalize(proj_emb, dim=1)

        interaction_score = self.interaction_head(graph_emb)

        return proj_emb, interaction_score


def contrastive_loss(z1, z2, temperature=0.1):
    """Contrastive loss (InfoNCE) for self-supervised learning"""
    batch_size = z1.shape[0]
    device = z1.device

    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    sim_matrix = torch.matmul(z1, z2.T) / temperature
    labels = torch.arange(batch_size, device=device)

    loss = F.cross_entropy(sim_matrix, labels)
    loss_reverse = F.cross_entropy(sim_matrix.T, labels)

    return (loss + loss_reverse) / 2.0


def train_contrastive_model(model, graphs, device, epochs=10, batch_size=32, lr=0.01, gradient_accumulation_steps=4):
    """Train model with self-supervised contrastive learning"""
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    losses = []
    target_channels = max([g.x.shape[1] for g in graphs]) if len(graphs) > 0 else 4

    print(f"\n{'='*60}")
    print(f"Training Contrastive GAT Model")
    print(f"{'='*60}")
    print(f"  Epochs: {epochs}, Batch size: {batch_size}, LR: {lr}")
    print(f"  Total graphs: {len(graphs)}, Target channels: {target_channels}")

    if device.type == 'cuda':
        torch.cuda.empty_cache()

    for epoch in range(epochs):
        epoch_losses = []
        optimizer.zero_grad()

        shuffled_graphs = graphs.copy()
        random.shuffle(shuffled_graphs)

        for batch_idx, i in enumerate(range(0, len(shuffled_graphs), batch_size)):
            batch_graphs = shuffled_graphs[i:i+batch_size]

            aug1_graphs = [graph_augment(g, target_channels=target_channels) for g in batch_graphs]
            aug2_graphs = [graph_augment(g, target_channels=target_channels) for g in batch_graphs]

            for g in aug1_graphs + aug2_graphs:
                g.x = g.x.to(device)
                g.edge_index = g.edge_index.to(device)
                if hasattr(g, 'edge_attr') and g.edge_attr is not None:
                    g.edge_attr = g.edge_attr.to(device)

            try:
                batch1 = Batch.from_data_list(aug1_graphs)
                batch2 = Batch.from_data_list(aug2_graphs)
            except Exception as e:
                print(f"Error creating batch: {e}")
                continue

            z1, pred1 = model(batch1.x, batch1.edge_index, getattr(batch1, "edge_attr", None), batch1.batch)
            z2, _ = model(batch2.x, batch2.edge_index, getattr(batch2, "edge_attr", None), batch2.batch)

            # Contrastive loss
            loss_contrast = contrastive_loss(z1, z2, temperature=0.1)

            # Supervised regression loss on gt_score (if available)
            loss_reg = torch.tensor(0.0, device=device)
            if hasattr(batch1, "gt_score"):
                try:
                    gt_scores = batch1.gt_score.to(device).float()
                    pred_scores = pred1.view(-1)
                    # Normalize scores for stability
                    if gt_scores.std() > 0:
                        gt_scores = (gt_scores - gt_scores.mean()) / (gt_scores.std() + 1e-8)
                    if pred_scores.std() > 0:
                        pred_scores = (pred_scores - pred_scores.mean()) / (pred_scores.std() + 1e-8)
                    loss_reg = F.mse_loss(pred_scores, gt_scores)
                except:
                    pass

            # Combined loss
            loss = (loss_contrast + 0.5 * loss_reg) / gradient_accumulation_steps
            loss.backward()

            del z1, z2, batch1, batch2, aug1_graphs, aug2_graphs
            if device.type == 'cuda':
                torch.cuda.empty_cache()

            if (batch_idx + 1) % gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

            epoch_losses.append(loss.item() * gradient_accumulation_steps)

        if len(epoch_losses) > 0 and (len(shuffled_graphs) // batch_size) % gradient_accumulation_steps != 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        avg_loss = np.mean(epoch_losses) if len(epoch_losses) > 0 else 0.0
        losses.append(avg_loss)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.6f}")

    print(f"\n✓ Training complete!")
    return model, losses


# ============================================================================
# 4. Extract Coordinates using Model (predictions)
# ============================================================================

def find_max_interaction_positions(model, graphs, device, top_k=100):
    """
    Find positions with maximum interaction scores using the trained model.
    Uses the model's interaction_score, not any hand-crafted heuristic.
    """
    model.eval()
    all_scores = []

    target_channels = max([g.x.shape[1] for g in graphs]) if len(graphs) > 0 else 4

    with torch.no_grad():
        for graph in tqdm(graphs, desc="Processing graphs (model)"):
            # Prepare node features to have consistent channel dimension
            x = graph.x.clone()
            current_channels = x.shape[1]
            if current_channels < target_channels:
                padding = torch.zeros(x.shape[0], target_channels - current_channels, dtype=x.dtype, device=x.device)
                x = torch.cat([x, padding], dim=1)
            elif current_channels > target_channels:
                x = x[:, :target_channels]

            x = x.to(device)
            edge_index = graph.edge_index.to(device)
            if hasattr(graph, 'edge_attr') and graph.edge_attr is not None:
                edge_attr = graph.edge_attr.to(device)
            else:
                edge_attr = None

            # Forward pass: get model interaction score for this graph
            _, interaction_score = model(x, edge_index, edge_attr, batch=None)
            model_score = interaction_score.item()

            center = graph.center
            x_pos, y_pos, z_pos = center

            # Optional extra info for analysis
            num_nodes = graph.x.shape[0]
            num_edges = graph.edge_index.shape[1]
            num_channels = graph.x.shape[1] if hasattr(graph, 'x') else 0
            edge_density = num_edges / num_nodes if num_nodes > 0 else 0

            all_scores.append({
                'x': x_pos,
                'y': y_pos,
                'z': z_pos,
                'num_nodes': num_nodes,
                'num_edges': num_edges,
                'num_channels': num_channels,
                'edge_density': edge_density,
                'model_score': model_score
            })

    if len(all_scores) > 0:
        # Sort by model_score (descending): higher score = more important
        all_scores.sort(key=lambda x: x['model_score'], reverse=True)
        top_positions = all_scores[:top_k]
        return top_positions

    return []


# ============================================================================
# 5. Extract Ground-Truth Coordinates from Data (Option A)
# ============================================================================

def attach_gt_scores_to_graphs(data, graphs):
    """
    Compute a ground-truth score per graph center (same definition
    as extract_gt_coordinates_from_data) and store it as graph.gt_score.
    """
    num_channels, num_values, z_dim, y_dim, x_dim = data.shape

    # pattern_mask[z, y, x] = True if any channel / value > 0 at that voxel
    pattern_mask = (data > 0)          # (C, V, Z, Y, X)
    pattern_mask = pattern_mask.any(axis=1)  # (C, Z, Y, X)
    pattern_mask = pattern_mask.any(axis=0)  # (Z, Y, X)

    radius_sq = MAX_RADIUS ** 2

    for g in graphs:
        cx, cy, cz = g.center
        cx, cy, cz = int(cx), int(cy), int(cz)
        cz = max(0, min(z_dim - 1, cz))

        x_min = max(0, cx - MAX_RADIUS)
        x_max = min(x_dim, cx + MAX_RADIUS + 1)
        y_min = max(0, cy - MAX_RADIUS)
        y_max = min(y_dim, cy + MAX_RADIUS + 1)

        score = 0
        for y in range(y_min, y_max):
            dy2 = (y - cy) * (y - cy)
            for x in range(x_min, x_max):
                dx2 = (x - cx) * (x - cx)
                if dx2 + dy2 <= radius_sq:
                    if pattern_mask[cz, y, x]:
                        score += 1

        g.gt_score = float(score)

    return graphs


def extract_gt_coordinates_from_data(data, graphs, top_k=100):
    """
    Extract top coordinates from TRUE ground truth using Option A:
    A voxel is 'pattern' if ANY channel has a nonzero value.

    For each graph center, count how many pattern voxels are within radius MAX_RADIUS.
    Then pick the top_k centers with highest counts.
    """
    print("\nComputing ground-truth coordinates from data (Option A: ANY channel nonzero)...")

    num_channels, num_values, z_dim, y_dim, x_dim = data.shape

    # pattern_mask[z, y, x] = True if any channel / value > 0 at that voxel
    # data shape: (C, V, Z, Y, X)
    pattern_mask = (data > 0)        # bool (C, V, Z, Y, X)
    pattern_mask = pattern_mask.any(axis=1)  # any over values -> (C, Z, Y, X)
    pattern_mask = pattern_mask.any(axis=0)  # any over channels -> (Z, Y, X)

    all_scores = []
    radius_sq = MAX_RADIUS ** 2

    for graph in tqdm(graphs, desc="Processing graphs (GT)"):
        cx, cy, cz = graph.center
        cx, cy, cz = int(cx), int(cy), int(cz)

        # Safety clamp
        cz = max(0, min(z_dim - 1, cz))

        x_min = max(0, cx - MAX_RADIUS)
        x_max = min(x_dim, cx + MAX_RADIUS + 1)
        y_min = max(0, cy - MAX_RADIUS)
        y_max = min(y_dim, cy + MAX_RADIUS + 1)

        score = 0
        for y in range(y_min, y_max):
            dy2 = (y - cy) * (y - cy)
            for x in range(x_min, x_max):
                dx2 = (x - cx) * (x - cx)
                if dx2 + dy2 <= radius_sq:
                    if pattern_mask[cz, y, x]:
                        score += 1

        all_scores.append({
            'x': cx,
            'y': cy,
            'z': cz,
            'score': score
        })

    if len(all_scores) > 0:
        # Sort by true pattern count (descending)
        all_scores.sort(key=lambda s: s['score'], reverse=True)
        top_positions = all_scores[:top_k]
        return top_positions

    return []


# ============================================================================
# 6. Compute Accuracy (+ radius-aware IoU "Accuracy" column)
# ============================================================================

def compute_accuracy(model_coords, gt_coords, tol=MAX_RADIUS):
    """
    Compute accuracy metrics between model and ground truth coordinates.
    Radius-aware (tolerant) matching:
      - A model center is considered a match if it is within 'tol' distance of
        some *unmatched* GT center.
    Then:
      - precision  = matches / #model_points
      - recall     = matches / #gt_points
      - IoU (Accuracy) = matches / (model_points + gt_points - matches)
    """
    # Convert to simple lists of (x, y, z)
    model_points = [(int(c['x']), int(c['y']), int(c['z'])) for c in model_coords]
    gt_points    = [(int(c['x']), int(c['y']), int(c['z'])) for c in gt_coords]

    model_count = len(model_points)
    gt_count    = len(gt_points)

    if model_count == 0 or gt_count == 0:
        return {
            'matches': 0,
            'precision': 0.0,
            'recall': 0.0,
            'f1_score': 0.0,
            'model_count': model_count,
            'gt_count': gt_count,
            'accuracy_iou': 0.0,
        }

    tol_sq = tol ** 2
    matched_gt = set()
    matches = 0

    # greedy 1-to-1 matching within tolerance radius
    for mx, my, mz in model_points:
        for gi, (gx, gy, gz) in enumerate(gt_points):
            if gi in matched_gt:
                continue
            dx = mx - gx
            dy = my - gy
            dz = mz - gz
            dist_sq = dx*dx + dy*dy + dz*dz
            if dist_sq <= tol_sq:
                matches += 1
                matched_gt.add(gi)
                break

    precision = matches / model_count if model_count > 0 else 0.0
    recall    = matches / gt_count    if gt_count > 0 else 0.0
    f1_score  = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    union = model_count + gt_count - matches
    accuracy_iou = matches / union if union > 0 else 0.0

    return {
        'matches': matches,
        'precision': precision,
        'recall': recall,
        'f1_score': f1_score,
        'model_count': model_count,
        'gt_count': gt_count,
        'accuracy_iou': accuracy_iou,
    }


# ============================================================================
# 7. Main Execution Loop
# ============================================================================

if __name__ == "__main__":
    print("="*70)
    print("TEXTURE EXPERIMENT: Complete Pipeline (Radius-aware IoU)")
    print("="*70)
    print("\nThis will:")
    print("  1. Create 3 different textures for ground truth")
    print("  2. Visualize each ground truth (colorful plots)")
    print("  3. Generate subgraphs for each texture")
    print("  4. Train model for each texture")
    print("  5. Extract coordinates from model and GT")
    print("  6. Compute accuracy with radius-aware IoU")
    print("="*70)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nDevice: {device}\n")

    results = []

    # Process each texture
    for texture_type in TEXTURES:
        print(f"\n{'#'*70}")
        print(f"PROCESSING TEXTURE: {texture_type}")
        print(f"{'#'*70}\n")

        # 1. Create ground truth data with texture
        data, gt_filepath = create_groundtruth(texture_type, BASE_DIR)

        # 1b. Visualize ground truth (colorful plot)
        print(f"\n  Creating visualization...")
        vis_filepath = visualize_groundtruth(data, texture_type, BASE_DIR, z_slice=Z_FIXED)

        # 2. Create subgraphs
        graphs, subgraphs_filepath = create_subgraphs(data, texture_type, BASE_DIR)

        # 3. Filter graphs
        min_nodes = 100
        filtered_graphs = [g for g in graphs if g.x.shape[0] >= min_nodes]
        print(f"\n  Filtered to {len(filtered_graphs)} graphs (min {min_nodes} nodes)")

        if len(filtered_graphs) == 0:
            print(f"  ⚠ No graphs after filtering! Skipping {texture_type}...")
            continue

        # 4. Attach GT scores to graphs for supervised training
        print(f"\n  Attaching GT scores to graphs...")
        filtered_graphs = attach_gt_scores_to_graphs(data, filtered_graphs)

        # 5. Train model
        print(f"\n  Training model for {texture_type}...")
        in_channels = max([g.x.shape[1] for g in filtered_graphs])
        edge_dim = 1 if any(hasattr(g, 'edge_attr') and g.edge_attr is not None for g in filtered_graphs) else None

        model = ContrastiveGAT(
            in_channels=in_channels,
            hidden_channels=32,
            projection_dim=16,
            num_heads=4,
            dropout=0.1,
            edge_dim=edge_dim
        ).to(device)

        subsampled_graphs = filtered_graphs[::2] if len(filtered_graphs) > 20000 else filtered_graphs
        model, losses = train_contrastive_model(
            model=model,
            graphs=subsampled_graphs,
            device=device,
            epochs=10,  # Increased for better learning
            batch_size=32,
            lr=0.01,  # Reduced for more stable training
            gradient_accumulation_steps=4
        )

        # Save model
        model_filepath = os.path.join(BASE_DIR, f'model_{texture_type}.pt')
        model_info = {
            'model_state_dict': model.state_dict(),
            'in_channels': in_channels,
            'hidden_channels': 32,
            'projection_dim': 16,
            'num_heads': 4,
            'dropout': 0.1,
            'edge_dim': edge_dim,
            'losses': losses
        }
        torch.save(model_info, model_filepath)
        print(f"  ✓ Saved model to: {model_filepath}")

        # 6. Run GAT model to extract top positions (reconstruct and find top positions)
        print(f"\n  Running GAT model to extract top positions...")
        model_coords = find_max_interaction_positions(model, filtered_graphs, device, top_k=TOP_K)

        # Save model_coords as pickle (for internal use)
        model_coords_filepath = os.path.join(BASE_DIR, f'model_coords_{texture_type}.pkl')
        with open(model_coords_filepath, 'wb') as f:
            pickle.dump(model_coords, f)
        print(f"  ✓ Saved model coordinates to: {model_coords_filepath}")

        # Save top_positions_result as pickle
        top_positions_pkl_filepath = os.path.join(BASE_DIR, f'top_positions_result_{texture_type}.pkl')
        with open(top_positions_pkl_filepath, 'wb') as f:
            pickle.dump(model_coords, f)
        print(f"  ✓ Saved top positions result (pkl) to: {top_positions_pkl_filepath}")

        # Save top_positions_result as numpy array (extract x, y, z coordinates)
        top_positions_array = np.array([[c['x'], c['y'], c['z']] for c in model_coords], dtype=np.int32)
        top_positions_npy_filepath = os.path.join(BASE_DIR, f'top_positions_result_{texture_type}.npy')
        np.save(top_positions_npy_filepath, top_positions_array)
        print(f"  ✓ Saved top positions result (npy) to: {top_positions_npy_filepath}")

        # 7. Extract ground-truth coordinates from data (Option A)
        # IMPORTANT: Use filtered_graphs (same as model training) for fair comparison
        print(f"\n  Extracting coordinates from ground truth (data, using filtered_graphs)...")
        gt_coords = extract_gt_coordinates_from_data(data, filtered_graphs, top_k=TOP_K)
        gt_coords_filepath = os.path.join(BASE_DIR, f'gt_coords_{texture_type}.pkl')
        with open(gt_coords_filepath, 'wb') as f:
            pickle.dump(gt_coords, f)
        print(f"  ✓ Saved GT coordinates to: {gt_coords_filepath}")

        # 8. Compute accuracy (radius-aware)
        print(f"\n  Computing accuracy (radius-aware IoU, tol = {MAX_RADIUS})...")
        accuracy = compute_accuracy(model_coords, gt_coords, tol=MAX_RADIUS)
        results.append({
            'Texture': texture_type,
            'GT File': os.path.basename(gt_filepath),
            'Matches': accuracy['matches'],
            'Model Points': accuracy['model_count'],
            'GT Points': accuracy['gt_count'],
            'Precision': f"{accuracy['precision']:.4f}",
            'Recall': f"{accuracy['recall']:.4f}",
            'F1 Score': f"{accuracy['f1_score']:.4f}",
            'Accuracy': f"{accuracy['accuracy_iou']:.4f}",  # IoU with tolerance
        })
        print(
            f"  ✓ Accuracy: "
            f"Precision={accuracy['precision']:.4f}, "
            f"Recall={accuracy['recall']:.4f}, "
            f"F1={accuracy['f1_score']:.4f}, "
            f"IoU-Acc={accuracy['accuracy_iou']:.4f}"
        )

        # End of texture processing
        print(f"\n{'='*70}")
        print(f"Completed processing {texture_type}")
        print(f"{'='*70}\n")

    # 9. Create final results table (all textures)
    print(f"\n{'='*70}")
    print("FINAL RESULTS TABLE (All Textures)")
    print(f"{'='*70}\n")

    results_df = pd.DataFrame(results)
    print(results_df.to_string(index=False))

    # Save results table
    results_filepath = os.path.join(BASE_DIR, 'accuracy_results_all_textures.csv')
    results_df.to_csv(results_filepath, index=False)
    print(f"\n✓ Results saved to: {results_filepath}")

    print(f"\n{'='*70}")
    print("EXPERIMENT COMPLETE!")
    print(f"{'='*70}\n")
